# Import Libraries

In [68]:
import os
import time
import pickle
import random
import itertools

# Analisi dati e Visualizzazione
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn: Preprocessing e Dimensionality Reduction
from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
    LabelEncoder,
    OneHotEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.utils.class_weight import compute_class_weight

# Scikit-learn: Model Selection e Pipeline
from sklearn.model_selection import (
    train_test_split,
    PredefinedSplit,
    GridSearchCV,
    RandomizedSearchCV,
    ParameterGrid
)
from sklearn.pipeline import Pipeline

# Scikit-learn: Modelli di Machine Learning Classico
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import plot_tree

# Imbalanced Learning (SMOTE e Pipeline compatibile)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Metriche di Valutazione
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_auc_score,
    roc_curve,
    PrecisionRecallDisplay
)

# Deep Learning (PyTorch) e Monitoraggio
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

import datetime

# Our packages
from utils import *
from plot import *

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer


from sklearn.compose import make_column_selector
from sklearn import set_config

set_config(transform_output="pandas")

# Global Variables

In [69]:
SEED = 42
FILENAME = "../../data/train.csv"

# Cerca la GPU
if torch.backends.mps.is_available():
    print("MPS device is available.")
    device = torch.device("mps")
elif torch.cuda.is_available():
    print("CUDA device is available.")
    device = torch.device("cuda")
else:
    print("No GPU acceleration available.")
    device = torch.device("cpu")

No GPU acceleration available.


In [70]:
def fix_random(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

fix_random(SEED)

# Load the dataset

In [71]:
df = pd.read_csv(FILENAME, encoding='ISO-8859-1', sep=",")

rows = df.shape[0]
cols = df.shape[1]
print("# Righe: " + str(rows)+ " # Colonne: "+str(cols) + "\n")

# Righe: 148301 # Colonne: 145



# Preprocessing

## 1. Remove duplicates rows and columns

In [72]:
# Individua se esistono colonne con lo stesso nome
# Se esistono, allora se le colonne sono duplicati perfetti, droppiamo il duplicato
# Se esistono, ma nono sono perfetti duplicati, per intervenire consciamente sarebbe necessario avere maggior domain knowledge
feature_list = df.columns.to_list()
has_duplicate_cols = len(feature_list) != len(set(feature_list))
print("Ci sono colonne con lo stesso nome?", has_duplicate_cols)

if has_duplicate_cols:
    df2 = df.T.drop_duplicates().T


# Rimuovi righe duplicate
df.drop_duplicates(inplace=True)


##################################################
print("Nuovo # Righe: " + str(rows)+ " Nuovo # Colonne: "+str(cols) + "\n")


Ci sono colonne con lo stesso nome? False
Nuovo # Righe: 148301 Nuovo # Colonne: 145



## 2. Label extraction and train-test splitting

In [73]:
X = df.drop(columns=["grade"])
y = df["grade"]

In [74]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)

## 3. Pipeline

In [75]:
class ColumnDropper(BaseEstimator, TransformerMixin):
    """ Drop generic columns """
    def __init__(self, columns=[]):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        return X.drop(columns=[col for col in self.columns if col in X.columns])
    

class HighNanDropper(BaseEstimator, TransformerMixin):
    """ Rimuove colonne con alto numero di NaN"""
    
    def __init__(self, threshold=90):
        self.threshold = threshold
        self.columns = []

    def fit(self, X, y=None):
        nan_ratio = X.isna().mean() * 100
        self.columns = nan_ratio[nan_ratio > self.threshold].index.tolist()
        return self

    def transform(self, X):
        X = X.copy()
        return X.drop(columns=[col for col in self.columns if col in X.columns])
    

class HighlyCorrelatedDropper(BaseEstimator, TransformerMixin):
    """" Rimozione delle feature ridondanti identificate (High Correlation > threshold) """
    def __init__(self, threshold=0.95):
        self.threshold = threshold
        self.columns = []

    def fit(self, X, y=None):
        numeric_df = X.select_dtypes(include=[np.number])
        corr_matrix = numeric_df.corr().abs()
        # Selezioniamo il triangolo superiore della matrice; k=1 esclude la diagonale principale
        upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        self.columns = [column for column in upper_tri.columns if any(upper_tri[column] > self.threshold)]
        return self

    def transform(self, X):
        X = X.copy()
        print(self.columns)
        return X.drop(columns=[col for col in self.columns if col in X.columns])
    

class NumericExtractor(BaseEstimator, TransformerMixin):
    """ Estrae feature numeriche da stringhe (36/60 mesi, polishing anni di carriera...) """
    def __init__(self, columns=[]):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        valid_cols = [col for col in self.columns if col in X.columns]
        for col in valid_cols:
            X[col] = (X[col].astype(str)
                        .replace('< 1', '0') 
                        .str.extract(r"(\d+)")
                        .astype(float))
        return X
    
    
class FeatureAverager(BaseEstimator, TransformerMixin):
    """Media di n colonne e rimuove gli originali"""
    def __init__(self, columns=[], new_name='new_avg_col'):
        self.columns = columns
        self.new_name = new_name

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        # Calcoliamo la media lungo l'asse delle righe (axis=1)
        valid_cols = [col for col in self.columns if col in X.columns]
        if valid_cols:
            X[self.new_name] = X[valid_cols].mean(axis=1)
        return X.drop(columns=[col for col in self.columns if col in X.columns])

    
class DateDifferenceTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, reference_col, target_cols=None):
        self.reference_col = reference_col
        self.target_cols = target_cols or []
        self.date_format = '%b-%Y'

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        
        # Check if reference exists
        if self.reference_col not in X.columns:
            return X # Skip if ref is missing (maybe dropped by NaN filter)

        ref_series = pd.to_datetime(X[self.reference_col], format=self.date_format, errors='coerce')
        
        # Ensure targets is a list and filter for existence
        targets = [self.target_cols] if isinstance(self.target_cols, str) else self.target_cols
        valid_targets = [col for col in targets if col in X.columns]
        
        cols_to_drop = []

        for col in valid_targets:
            target_series = pd.to_datetime(X[col], format=self.date_format, errors='coerce')
            new_col_name = f"months_since_{col}"

            X[new_col_name] = (
                (ref_series.dt.year - target_series.dt.year) * 12 +
                (ref_series.dt.month - target_series.dt.month)
            )
            X[new_col_name] = np.round(X[new_col_name]).astype('Int64')
            cols_to_drop.append(col)

        # Drop columns
        all_drops = cols_to_drop + [self.reference_col]
        # Final safety check before drop
        X = X.drop(columns=[c for c in all_drops if c in X.columns], errors='ignore')

        return X
    

class RoundToIntTransformer(BaseEstimator, TransformerMixin):
    """ Arrotonda le colonne selezionate all'intero piu vicino """
    def __init__(self, columns=[]):
        self.columns = columns
    
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col in [c for c in self.columns if c in X.columns]:
                X[col] =  np.round(X[col]).astype('Int64')
        return X
    



In [76]:
loan_performance_data_leakage = [
    'loan_status_current_code',                         # prestito in regola, in ritardo, totalmente pagato...
    'outstanding_principal_balance',                    # "outstanding principal" e' la parte del capitale da restituire
    'outstanding_principal_investor_side',              # similmente
    'total_payment_received',                           # somma pagata al creditore
    'total_payment_investor_side',
    'total_received_principal',                         # somma pagata al creditore che copre la il capitale del prestito
    'total_received_interest',                          # ... copre gli interessi
    'total_received_late_fees',                         # ... copre le penali
    'recoveries_cash',                                  # somma recuperata dopo un prestito andato in default
    'collection_recovery_fee',                          # spese per il recupero crediti
    'last_payment_date',                                # data ultimo pagamento effettuato
    'last_payment',                                     # importo ultimo pagamento
    'next_payment_date',                                # data prossimo pagamento
    'last_credit_pull_date',                            # data ultimo check profilo creditizio, durante il periodo di prestito
    'last_fico_score_high_bound',                       # ultimo punteggio FICO rilevato: al momento della concessione del prestito si usa 'fico_score_low_bound', 'fico_score_high_bound
    'last_fico_score_low_bound',
    'total_collection_amount',
    'loan_payment_installments_count'                   # potrebbe semprare il numero di rate, ma la tipologia di valori contenuti fa pensare al valore economico della singola rata (calcolo derivante di interest rate)
]
# "Settlement" indica una situazione avvenuta durante / dopo il prestito, non al momento della concessione
settlement_data_leakage = [col for col in X.columns if 'settlement' in col]
# "Hardship loans" sono concessioni per agevolare il pagamento di un prestito quando il debitore si trova in momenti di difficoltà economica (perdita lavoro, problemi medici, disastri naturali)
# https://www.oaic.gov.au/privacy/your-privacy-rights/credit-reporting/hardship-assistance/what-is-a-financial-hardship-arrangement
# https://financialrights.org.au/factsheet/financial-hardship/
hardship_data_leakage = [col for col in X.columns if 'hardship' in col]
other_leakage = [
    'original_projected_additional_accrued_interest',           # interesse addizionale previsto, presumibilmente in seguito a modifiche di piani ammortamento o hardship
    #'loan_issue_date',                                         # Il grade è influenzato dalla situazione creditizia del richiedente, più che dal periodo
                                                                # droppato in un secondo momento, dopo averlo usato per feature extraction
    'investor_side_funded_amount',
    'loan_portfolio_total_funded',
]
# Il tasso di interesse di un prestito è calcolato basandosi sul Grading assegnato al prestito stesso.
# Essendo una conseguenza del nostro target "grade", è da considerarsi data leakage
# https://www.airtel.in/blog/personal-loan/how-does-loan-grading-work/ (Accessed 02/02/2026)
loan_contract_interest_rate = [
    'loan_contract_interest_rate'
]
other_non_significant = [
    'platform_policy_code_id',                                      # id interno al prestatore
    'loan_title',                                                   # non significant column, grande sparsita' di dati. Sufficiente loan_purpose_category come aggregazione di scopo del prestito
    'borrower_address_zip',                                         # non significant column, esiste una colonna per identificazione stati
]

joint_and_secondary_cols = [
    'joint_income_annual',
    'joint_dti_ratio',
    'joint_income_verification_status',
    'joint_revolving_balance',
    'secondary_applicant_fico_low',
    'secondary_applicant_fico_high',
    'secondary_applicant_earliest_credit_line',
    'secondary_applicant_inquiries_6m',
    'secondary_applicant_mortgage_accounts',
    'secondary_applicant_open_accounts',
    'secondary_applicant_revolving_utilization',
    'secondary_applicant_open_active_installment_loans',
    'secondary_applicant_revolving_accounts',
    'secondary_applicant_chargeoffs_12m',
    'secondary_applicant_collections_12m_ex_med',
    'secondary_applicant_months_since_last_major_derog',
]

number_from_string_cols = [
    'loan_contract_term_months',
    'borrower_profile_employment_length',
]

average_cols = [
    'fico_score_low_bound',
    'fico_score_high_bound'
]

date_diff_reference = 'loan_issue_date'
date_diff_target = [ 'credit_history_earliest_line' ]


# Lista delle feature che richiedono valori interi: conteggi di linee di credito e conti, mesi, fico score
round_to_nearest_int = [
    'loan_contract_term_months', 'credit_delinquencies_2yrs', 'credit_inquiries_6m',
    'months_since_last_delinquency', 'months_since_last_public_record', 
    'credit_open_accounts', 'credit_public_records', 'credit_total_accounts',
    'collections_12m_ex_med', 'months_since_last_major_derog', 'accounts_now_delinquent',
    'open_accounts_6m', 'open_active_installment_loans', 'open_installment_loans_12m',
    'open_installment_loans_24m', 'months_since_recent_installment_loan', 
    'open_revolving_accounts_12m', 'open_revolving_accounts_24m', 'finance_inquiries',
    'credit_union_trades_total', 'credit_inquiries_12m', 'accounts_open_past_24m',
    'chargeoffs_within_12m', 'months_since_oldest_installment_acct', 
    'months_since_oldest_revolving_acct', 'months_since_recent_revolving_acct',
    'months_since_recent_trade_line', 'mortgage_accounts', 'months_since_recent_bankcard',
    'months_since_recent_bankcard_delinquency', 'months_spen_installment_loans_12m',
    'open_installment_loans_24m', 'months_since_recent_installment_loan', 
    'open_revolving_accounts_12m', 'open_revolving_accounts_24m', 'finance_inquiries',
    'credit_union_trades_total', 'credit_inquiries_12m', 'accounts_open_past_24m',
    'chargeoffs_within_12m', 'months_since_oldest_installment_acct', 
    'months_since_oldest_revolving_acct', 'months_since_recent_revolving_acct',
    'months_since_recent_trade_line', 'mortgage_accounts', 'months_since_recent_bankcard',
    'months_since_recent_bankcard_delinquency', 'months_since_recent_inquiry',
    'months_since_recent_revolving_delinquency', 'accounts_ever_120dpd',
    'active_bankcard_tradelines', 'active_revolving_tradelines', 
    'bankcard_satisfactory_accounts', 'bankcard_tradelines', 'installment_tradelines',
    'open_revolving_tradelines', 'revolving_accounts', 'tradelines_120dpd_2m',
    'tradelines_30dpd', 'tradelines_90dpd_24m', 'tradelines_open_past_12m',
    'public_record_bankruptcies', 'tax_liens_total', 'fico_average', 
    'months_since_credit_history_earliest_line' # newly created
]

categorical_to_unknown_cols = [
  'borrower_address_state',
  'loan_purpose_category',
  'borrower_income_verification_status',
  'borrower_housing_ownership_status'
]

fill_big_cols = [
    'months_since_last_public_record',
    'months_since_recent_bankcard_delinquency',
    'months_since_last_major_derog',
    'months_since_recent_revolving_delinquency',
    'months_since_last_delinquency', 
    'months_since_recent_inquiry', 
    'months_since_recent_bankcard',
    'months_since_recent_trade_line'
]

fill_zero_cols = [
    'open_accounts_6m', 'open_installment_loans_12m', 'open_installment_loans_24m',
    'open_revolving_accounts_12m', 'open_revolving_accounts_24m', 'finance_inquiries',
    'credit_inquiries_12m', 'delinquency_amount', 
    'tax_liens_total', 'mortgage_accounts', 'chargeoffs_within_12m', 
    'collections_12m_ex_med', 'accounts_now_delinquent', 'public_record_bankruptcies',
    'credit_public_records', 'credit_delinquencies_2yrs',
    'borrower_profile_employment_length',

    'tradelines_120dpd_2m', 'tradelines_30dpd', 'tradelines_90dpd_24m', 
    'accounts_ever_120dpd', 'credit_union_trades_total', 'open_active_installment_loans', 
    'credit_inquiries_6m', 'tradelines_open_past_12m', 'accounts_open_past_24m',
    'open_revolving_tradelines', 'active_bankcard_tradelines', 'installment_tradelines',
    'revolving_accounts', 'bankcard_satisfactory_accounts',
    'active_revolving_tradelines',  'bankcard_tradelines',
    'credit_open_accounts', 'credit_total_accounts'
]

fill_to_mode_cat = [
    'disbursement_method_type',
    'application_type_label',
    'listing_initial_status',
    'loan_payment_plan_flag'         # n/y
]

fill_to_mode_num = [
    'loan_contract_term_months'    # 36/60 mesi
]

In [77]:
def existing(cols, df):
    return [c for c in cols if c in df.columns]

remainder_pipeline = Pipeline([
    ('impute_median', SimpleImputer(strategy='median'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('impute_unknown', SimpleImputer(strategy='constant', fill_value='Unknown'), lambda df: existing(categorical_to_unknown_cols, df)),
        ('impute_0', SimpleImputer(strategy='constant', fill_value=0), lambda df: existing(fill_zero_cols, df)),
        ('impute_10k', SimpleImputer(strategy='constant', fill_value=999), lambda df: existing(fill_big_cols, df)),
        ('impute_mode_cat',SimpleImputer(strategy='most_frequent'), lambda df: existing(fill_to_mode_cat, df)),
        ('impute_mode_num', SimpleImputer(strategy='most_frequent'), lambda df: existing(fill_to_mode_num, df)),
    ],
    remainder=remainder_pipeline,
    verbose_feature_names_out=False
)


encoding = ColumnTransformer(
    transformers=[
        # Applica categorical_pipe a TUTTE le colonne di tipo object o category
        ('cat_step', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), make_column_selector(dtype_include=['object'])),
    ], 
    # Tutto ciò che è numerico e non è stato toccato sopra, viene scalato qui
    remainder="passthrough", 
    verbose_feature_names_out=False
)

In [ ]:

full_pipeline = Pipeline([
    ('drop_leakage', ColumnDropper(columns = loan_performance_data_leakage + settlement_data_leakage + hardship_data_leakage + other_leakage + loan_contract_interest_rate)),
    ('drop_non_significant', ColumnDropper(columns = other_non_significant)),
    ('drop_joint_and_secondary', ColumnDropper(columns = joint_and_secondary_cols)),
    ('drop_high_nan', HighNanDropper(threshold=0.9)),
    ('high_correlation', HighlyCorrelatedDropper(threshold=0.95)),
    ('feature_extraction', NumericExtractor(columns = number_from_string_cols)),
    ('fico_average', FeatureAverager(columns = average_cols)),
    ('date_diff', DateDifferenceTransformer(reference_col=date_diff_reference, target_cols=date_diff_target)),
    ('rounding_int', RoundToIntTransformer(columns = round_to_nearest_int)),

    ('preprocessor', preprocessor),
    ('encoding', encoding),
    
    ("clf", RandomForestClassifier(random_state=SEED, class_weight='balanced'))
])

param_grid_rf = {
    'clf__n_estimators': [250, 300],
    'clf__criterion': ['gini', 'log_loss'],
    'clf__max_depth': [10, 20]
}

scoring = {
    'acc': 'accuracy',
    'balanced_acc': 'balanced_accuracy',
    'f1_weighted': 'f1_weighted'
}

grid = GridSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid_rf,
    cv=3,
    scoring='balanced_accuracy',
    #refit='balanced_acc', 
    error_score='raise',
    n_jobs=-1,
    verbose=3
)
grid.fit(X_train, y_train)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV 1/3] END clf__criterion=gini, clf__max_depth=10, clf__n_estimators=250;, score=0.363 total time=  16.3s
[CV 2/3] END clf__criterion=gini, clf__max_depth=10, clf__n_estimators=250;, score=0.359 total time=  16.1s
[CV 3/3] END clf__criterion=gini, clf__max_depth=10, clf__n_estimators=250;, score=0.361 total time=  16.3s
[CV 1/3] END clf__criterion=gini, clf__max_depth=10, clf__n_estimators=300;, score=0.362 total time=  18.9s
[CV 2/3] END clf__criterion=gini, clf__max_depth=10, clf__n_estimators=300;, score=0.360 total time=  19.4s
[CV 3/3] END clf__criterion=gini, clf__max_depth=10, clf__n_estimators=300;, score=0.362 total time=  19.2s
[CV 1/3] END clf__criterion=gini, clf__max_depth=20, clf__n_estimators=250;, score=0.352 total time=  31.1s
[CV 2/3] END clf__criterion=gini, clf__max_depth=20, clf__n_estimators=250;, score=0.359 total time=  30.7s
[CV 3/3] END clf__criterion=gini, clf__max_depth=20, clf__n_estimators=250;,

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'clf__criterion': ['gini', 'log_loss'], 'clf__max_depth': [10, 20], 'clf__n_estimators': [250, 300]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'balanced_accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candida

In [81]:
# Migliori stats trovate
print("Best Parameters:", grid.best_params_)
print("Best Estimator:", grid.best_estimator_)
print(f"Best CV score: {grid.best_score_:.4f}")

""" print("--"*20)
results = grid_search.cv_results_
best_index = grid_search.best_index_

# Now these keys will exist because you used the dictionary scoring
print(f"Best Model Results (Refit on F1_Weighted):")
print(f"Mean Accuracy: {results['mean_test_acc'][best_index]:.4f}")
print(f"Mean Balanced Acc: {results['mean_test_balanced_acc'][best_index]:.4f}")
print(f"Mean F1 Weighted: {results['mean_test_f1_weighted'][best_index]:.4f}")

print("--"*20) """

Best Parameters: {'clf__criterion': 'gini', 'clf__max_depth': 10, 'clf__n_estimators': 300}
Best Estimator: Pipeline(steps=[('drop_leakage',
                 ColumnDropper(columns=['loan_status_current_code',
                                        'outstanding_principal_balance',
                                        'outstanding_principal_investor_side',
                                        'total_payment_received',
                                        'total_payment_investor_side',
                                        'total_received_principal',
                                        'total_received_interest',
                                        'total_received_late_fees',
                                        'recoveries_cash',
                                        'collection_recovery_fee',
                                        'last_payment_date'...
                ('encoding',
                 ColumnTransformer(remainder='passthrough',
                     

' print("--"*20)\nresults = grid_search.cv_results_\nbest_index = grid_search.best_index_\n\n# Now these keys will exist because you used the dictionary scoring\nprint(f"Best Model Results (Refit on F1_Weighted):")\nprint(f"Mean Accuracy: {results[\'mean_test_acc\'][best_index]:.4f}")\nprint(f"Mean Balanced Acc: {results[\'mean_test_balanced_acc\'][best_index]:.4f}")\nprint(f"Mean F1 Weighted: {results[\'mean_test_f1_weighted\'][best_index]:.4f}")\n\nprint("--"*20) '

In [84]:
import pickle

# --- 1. Split the Best Estimator ---
full_best_pipeline = grid.best_estimator_

# Slicing [:-1] creates a new Pipeline containing ONLY the 'dropper' and 'prep' steps
# It retains all the fitted logic (modes, dropped columns, etc.)
preprocessing_part = full_best_pipeline[:-1] 

# Access the 'rf' step directly to get just the classifier
classifier_part = full_best_pipeline.named_steps['clf']

# --- 2. Save them as separate pickles ---
with open("rf_preprocessor.save", "wb") as f:
    pickle.dump(preprocessing_part, f)

with open("rf.save", "wb") as f:
    pickle.dump(grid.best_estimator_['clf'], f)

print("Saved 'rf_preprocessor.save' and 'rf_model.save' separately.")

PicklingError: Can't pickle <function <lambda> at 0x7efdab0004a0>: attribute lookup <lambda> on __main__ failed

In [ ]:

import pickle

# --- 1. Split the Best Estimator ---
full_best_pipeline = grid.best_estimator_

# Slicing [:-1] creates a new Pipeline containing ONLY the 'dropper' and 'prep' steps
# It retains all the fitted logic (modes, dropped columns, etc.)
preprocessing_part = full_best_pipeline[:-1] 

# Access the 'rf' step directly to get just the classifier
classifier_part = full_best_pipeline.named_steps['rf']

# --- 2. Save them as separate pickles ---
with open("svm_preprocessor.save", "wb") as f:
    pickle.dump(preprocessing_part, f)

with open("svm_model.save", "wb") as f:
    pickle.dump(classifier_part, f)

print("Saved 'svm_preprocessor.save' and 'svm_model.save' separately.")



In [ ]:
evaluate_model(X_val, y_val, "rf")